# Step 05 – Paired McNemar tests and spatial block cross-validation

Paper 2: *Benchmarking Ten Machine-Learning Algorithms for Mediterranean Forest Mapping with Sentinel-2* (Remote Sensing, 2026).

- **In the paper:** Sections 2.6, 3.2–3.3 · Table 6, Table 7, Figure 7, Tables S3 and S5, Figure S3
- **Needs:** steps 03 and 04
- **Writes:** `RESULTS_DIR/spatial_cv/`

Run the configuration cell below first.

In [ ]:
# ---------------------------------------------------------------- paths
# The data are not included in this repository. Set the two folders below to
# where the files are on your computer (or on your Google Drive in Colab).
#   DATA_DIR    : input data (training samples, Sentinel-2 stack, map tiles, rasters)
#   RESULTS_DIR : outputs written by the notebooks. Later steps read the outputs
#                 of earlier ones (e.g. RESULTS_DIR/train_test_gap/ from step 04).
import os
DATA_DIR    = 'data/'
RESULTS_DIR = 'results/'

# Google Colab: mount Drive and point the folders to it, for example
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR    = '/content/drive/MyDrive/<your_folder>/'
# RESULTS_DIR = '/content/drive/MyDrive/<your_folder>/results/'

os.makedirs(RESULTS_DIR, exist_ok=True)

# Files and folders used by this notebook:
#   DATA_DIR / S2_Features_For_Correlation_v3.csv
#   RESULTS_DIR / correlation_training_only/
#   RESULTS_DIR / spatial_cv/
#   RESULTS_DIR / train_test_gap/

In [ ]:
!pip install -q catboost

In [ ]:
"""
Spatial block cross-validation
+ exact McNemar tests with Holm correction on the common test set

Run AFTER step 04 (04_model_tuning_and_evaluation) (uses the ten tuned models saved there).
In Colab first:  !pip install -q catboost

SECONDARY ANALYSIS. The ten tuned configurations are re-evaluated with 5-fold
cross-validation on all 2,800 samples under the same conditions except one:
  random      : StratifiedKFold - points assigned to folds at random
  spatial_Xkm : StratifiedGroupKFold over square X km blocks - all points of a block
                are either in training or in testing
The accuracy drop (random - spatial) shows how much accuracy depends on spatial
proximity between training and test points. Both schemes use the same data, models,
fold count and predictors, so the comparison is like-for-like. The 16 predictors and
the hyperparameters come from the main analysis and are not re-tuned inside the folds
(state this in the Methods; it affects both schemes equally).

Outputs (OUT_DIR):
  spatial_cv_folds.csv              every model x scheme x fold (OA, kappa, macro F1)
  spatial_cv_summary.csv            mean +/- SD per model and scheme + accuracy drop
  spatial_cv_class_recall.csv       per-class recall, pooled out-of-fold, per scheme
  spatial_cv_design.csv             blocks, fold sizes, test-to-training distances
  spatial_cv_oof_predictions.csv    out-of-fold predictions (for maps / later checks)
  Fig_spatial_cv_accuracy.(png|pdf) random vs spatial OA per model
  Fig_spatial_cv_folds_<X>km.(png|pdf)  map of blocks coloured by fold
  mcnemar_pairs.csv, mcnemar_holm_matrix.csv   Table S1
"""


import os, json, time
from itertools import combinations
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from pyproj import Transformer
from scipy.spatial import cKDTree
from scipy.stats import binomtest
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score, recall_score

# ------------------------------------------------------------------ config
DATA_PATH  = DATA_DIR + 'S2_Features_For_Correlation_v3.csv'
SCREEN_DIR = RESULTS_DIR + 'correlation_training_only/'
MODEL_DIR  = RESULTS_DIR + 'train_test_gap/'
OUT_DIR    = RESULTS_DIR + 'spatial_cv/'
LABEL_COL  = 'class'
CRS        = 'EPSG:32629'
SEED       = 42
N_FOLDS    = 5
BLOCK_SIZES_KM = [10, 20]      # 10 km main result, 20 km sensitivity
MODELS = ['CatBoost', 'XGBoost', 'GTB', 'MLP', 'RF', 'SVM-RBF', 'KNN', 'CART', 'MDC', 'NB']
CLASS_NAMES = {0: 'Water', 1: 'Forest', 2: 'Bare ground', 3: 'Urban', 4: 'Crops/grass'}

os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------------ data and coordinates
PREDICTORS = pd.read_csv(SCREEN_DIR + 'v5_final_vif_training.csv')['feature'].tolist()
df = pd.read_csv(DATA_PATH)
X = df[PREDICTORS]
le = LabelEncoder().fit(df[LABEL_COL])
y = le.transform(df[LABEL_COL])
K = len(le.classes_)

def get_lonlat(frame):
    for lon, lat in [('longitude', 'latitude'), ('lon', 'lat')]:
        if lon in frame.columns and lat in frame.columns:
            return frame[[lon, lat]].to_numpy(dtype=float)
    if '.geo' in frame.columns:
        return np.array([json.loads(g)['coordinates'] for g in frame['.geo']], dtype=float)
    raise ValueError(f'No coordinates found. Columns: {frame.columns.tolist()}')

lonlat = get_lonlat(df)
xs, ys = Transformer.from_crs('EPSG:4326', CRS, always_xy=True).transform(lonlat[:, 0], lonlat[:, 1])
xy = np.column_stack([xs, ys])

# ------------------------------------------------------------------ fold schemes
def block_ids(size_km):
    s = size_km * 1000
    key = pd.Series(np.floor(xs / s).astype(int)).astype(str) + '_' + \
          pd.Series(np.floor(ys / s).astype(int)).astype(str)
    return pd.factorize(key)[0]

schemes, groups = {}, {}
schemes['random'] = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(X, y))
for b in BLOCK_SIZES_KM:
    g = block_ids(b)
    groups[b] = g
    schemes[f'spatial_{b}km'] = list(
        StratifiedGroupKFold(N_FOLDS, shuffle=True, random_state=SEED).split(X, y, g))

design = []
for name, folds in schemes.items():
    for k, (tr, te) in enumerate(folds):
        d, _ = cKDTree(xy[tr]).query(xy[te])       # distance to nearest training point
        missing = K - np.unique(y[tr]).size
        design.append({'scheme': name, 'fold': k, 'n_train': len(tr), 'n_test': len(te),
                       'min_class_in_test': int(np.bincount(y[te], minlength=K).min()),
                       'classes_missing_in_train': missing,
                       'nn_dist_median_m': np.median(d), 'nn_dist_p10_m': np.percentile(d, 10)})
        assert missing == 0, f'{name} fold {k}: a class is missing from training - change SEED'
design = pd.DataFrame(design).round(1)
design.to_csv(OUT_DIR + 'spatial_cv_design.csv', index=False)
for b in BLOCK_SIZES_KM:
    cnt = np.bincount(groups[b])
    print(f'{b} km blocks: {cnt.size} blocks with points | points per block: '
          f'median {np.median(cnt):.0f}, max {cnt.max()}')
print('\nDistance from test points to nearest training point (median over folds, m):')
print(design.groupby('scheme')[['nn_dist_median_m', 'nn_dist_p10_m', 'n_test', 'min_class_in_test']]
      .median().to_string())

# ------------------------------------------------------------------ cross-validation
def scores(yt, yp):
    return {'OA': accuracy_score(yt, yp), 'kappa': cohen_kappa_score(yt, yp),
            'F1': f1_score(yt, yp, average='macro')}

rows, recall_rows = [], []
oof = pd.DataFrame({'row_idx': df.index, 'y_true': le.classes_[y]})
for m in MODELS:
    base = joblib.load(MODEL_DIR + f'{m}.joblib')
    t0 = time.perf_counter()
    for name, folds in schemes.items():
        pred = np.full(len(y), -1)
        for k, (tr, te) in enumerate(folds):
            est = clone(base)
            est.fit(X.iloc[tr], y[tr])
            pred[te] = np.ravel(est.predict(X.iloc[te])).astype(int)
            rows.append({'Model': m, 'scheme': name, 'fold': k, **scores(y[te], pred[te])})
        oof[f'{m}__{name}'] = le.classes_[pred]
        rec = recall_score(y, pred, labels=range(K), average=None, zero_division=0)
        recall_rows += [{'Model': m, 'scheme': name, 'Class': CLASS_NAMES.get(c, str(c)),
                         'recall_pooled (%)': 100 * r} for c, r in zip(le.classes_, rec)]
    print(f'{m}: {time.perf_counter() - t0:.0f} s')

folds_df = pd.DataFrame(rows)
folds_df.to_csv(OUT_DIR + 'spatial_cv_folds.csv', index=False)
oof.to_csv(OUT_DIR + 'spatial_cv_oof_predictions.csv', index=False)
recall_df = pd.DataFrame(recall_rows).round(2)
recall_df.to_csv(OUT_DIR + 'spatial_cv_class_recall.csv', index=False)

agg = folds_df.groupby(['Model', 'scheme'])[['OA', 'kappa', 'F1']].agg(['mean', 'std'])
agg.columns = [f'{a}_{b}' for a, b in agg.columns]
agg = agg.reset_index()
summary = agg.pivot(index='Model', columns='scheme', values=['OA_mean', 'OA_std', 'kappa_mean', 'F1_mean'])
summary.columns = [f'{a}__{b}' for a, b in summary.columns]
for b in BLOCK_SIZES_KM:
    summary[f'OA_drop_pp__{b}km'] = 100 * (summary['OA_mean__random'] - summary[f'OA_mean__spatial_{b}km'])
summary = summary.sort_values('OA_mean__random', ascending=False)
summary.to_csv(OUT_DIR + 'spatial_cv_summary.csv')

show = pd.DataFrame({'Random CV OA (%)': [f'{100*a:.2f} ± {100*s:.2f}' for a, s in
                                          zip(summary['OA_mean__random'], summary['OA_std__random'])]},
                    index=summary.index)
for b in BLOCK_SIZES_KM:
    show[f'Spatial {b} km OA (%)'] = [f'{100*a:.2f} ± {100*s:.2f}' for a, s in
                                      zip(summary[f'OA_mean__spatial_{b}km'], summary[f'OA_std__spatial_{b}km'])]
    show[f'Drop {b} km (pp)'] = summary[f'OA_drop_pp__{b}km'].round(2)
show.to_csv(OUT_DIR + 'spatial_cv_table.csv')
print('\n=== Random vs spatial block CV ===\n', show.to_string())

main = f'spatial_{BLOCK_SIZES_KM[0]}km'
print(f'\n=== Per-class recall, random vs {main} (pooled out-of-fold) ===')
print(recall_df[recall_df['scheme'].isin(['random', main])]
      .pivot_table(index=['Model', 'Class'], columns='scheme', values='recall_pooled (%)')
      .loc[MODELS[:3]].round(1).to_string())

# ------------------------------------------------------------------ figures
fig, ax = plt.subplots(figsize=(7.1, 3.2))
order = summary.index.tolist()
cols = ['random'] + [f'spatial_{b}km' for b in BLOCK_SIZES_KM]
labels = ['Random CV'] + [f'Spatial block CV ({b} km)' for b in BLOCK_SIZES_KM]
colors = ['#0072B2', '#D55E00', '#E69F00']
w = 0.8 / len(cols)
for i, (c, lab, col) in enumerate(zip(cols, labels, colors)):
    ax.bar(np.arange(len(order)) + (i - (len(cols) - 1) / 2) * w,
           100 * summary.loc[order, f'OA_mean__{c}'], w, yerr=100 * summary.loc[order, f'OA_std__{c}'],
           color=col, label=lab, capsize=2, error_kw={'lw': 0.8})
ax.set_xticks(range(len(order)), order, rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Overall accuracy (%)', fontsize=9)
lo = 100 * min(summary[f'OA_mean__{c}'].min() for c in cols)
ax.set_ylim(max(0, lo - 10), 100)
ax.tick_params(labelsize=8)
ax.grid(axis='y', alpha=0.3, lw=0.5)
ax.legend(fontsize=7.5, frameon=False, ncol=3, loc='upper center', bbox_to_anchor=(0.5, 1.15))
fig.tight_layout()
fig.savefig(OUT_DIR + 'Fig_spatial_cv_accuracy.png', dpi=600, bbox_inches='tight')
fig.savefig(OUT_DIR + 'Fig_spatial_cv_accuracy.pdf', bbox_inches='tight')
plt.show()

for b in BLOCK_SIZES_KM:
    fold_of = np.empty(len(y), dtype=int)
    for k, (_, te) in enumerate(schemes[f'spatial_{b}km']):
        fold_of[te] = k
    fig, ax = plt.subplots(figsize=(5.2, 5.2))
    sc = ax.scatter(xs / 1000, ys / 1000, c=fold_of, cmap='tab10', vmin=-0.5, vmax=9.5, s=4)
    ax.set_aspect('equal')
    ax.set_xlabel('Easting (km)', fontsize=9); ax.set_ylabel('Northing (km)', fontsize=9)
    ax.tick_params(labelsize=8)
    ax.set_title(f'Reference points by test fold ({b} km blocks, {CRS})', fontsize=9)
    handles = [plt.Line2D([], [], marker='o', ls='', color=plt.cm.tab10(k), label=f'Fold {k + 1}')
               for k in range(N_FOLDS)]
    ax.legend(handles=handles, fontsize=7.5, frameon=False, loc='best')
    fig.tight_layout()
    fig.savefig(OUT_DIR + f'Fig_spatial_cv_folds_{b}km.png', dpi=600, bbox_inches='tight')
    fig.savefig(OUT_DIR + f'Fig_spatial_cv_folds_{b}km.pdf', bbox_inches='tight')
    plt.show()

# ------------------------------------------------------------------ Table S1: McNemar
tp = pd.read_csv(MODEL_DIR + 'test_predictions.csv')
yt = tp['y_true'].values
correct = {m: (tp[m].values == yt) for m in MODELS}
pairs = []
for a, b in combinations(MODELS, 2):
    n_a = int((correct[a] & ~correct[b]).sum())     # a correct, b wrong
    n_b = int((~correct[a] & correct[b]).sum())     # b correct, a wrong
    p = binomtest(min(n_a, n_b), n_a + n_b, 0.5).pvalue if n_a + n_b else 1.0
    pairs.append({'Model A': a, 'Model B': b, 'OA A (%)': 100 * correct[a].mean(),
                  'OA B (%)': 100 * correct[b].mean(), 'A right, B wrong': n_a,
                  'B right, A wrong': n_b, 'p_exact': p})
mc = pd.DataFrame(pairs)
order_p = np.argsort(mc['p_exact'].values)                 # Holm step-down
adj, running, n_tests = np.empty(len(mc)), 0.0, len(mc)
for rank, idx in enumerate(order_p):
    running = max(running, (n_tests - rank) * mc['p_exact'].values[idx])
    adj[idx] = min(1.0, running)
mc['p_holm'] = adj
mc['significant (0.05)'] = mc['p_holm'] < 0.05
mc = mc.round({'OA A (%)': 2, 'OA B (%)': 2, 'p_exact': 4, 'p_holm': 4})
mc.to_csv(OUT_DIR + 'mcnemar_pairs.csv', index=False)

mat = pd.DataFrame('', index=MODELS, columns=MODELS)
for _, r in mc.iterrows():
    mat.loc[r['Model B'], r['Model A']] = f"{r['p_holm']:.3f}" + ('*' if r['significant (0.05)'] else '')
mat.to_csv(OUT_DIR + 'mcnemar_holm_matrix.csv')
print(f"\n=== Table S1: McNemar exact test, Holm-adjusted p ({n_tests} pairs; * = p < 0.05) ===")
print(mat.to_string())
top = [m for m in MODELS if correct[m].mean() >= max(c.mean() for c in correct.values()) - 0.02]
sig_top = mc[mc['Model A'].isin(top) & mc['Model B'].isin(top) & mc['significant (0.05)']]
print(f'\nModels within 2 pp of the best: {top}')
print('Significant differences among them:', 'none' if sig_top.empty else
      sig_top[['Model A', 'Model B', 'p_holm']].to_string(index=False))
print('\nFiles saved to:', OUT_DIR)